In [20]:
import pandas as pd


In [21]:
# Read in RASMI material intensity data and MI from old IMAGE-Materials CP
# Residential housing


In [22]:
mi_rasmi = pd.read_excel("MI_ranges_20230905.xlsx", index_col=[0, 1, 2, 3, 4], 
                         sheet_name=["concrete", "brick", "wood", "steel", "glass", "plastics", "aluminum", "copper"])

mi_image_mat = pd.read_csv("Building_materials.csv", index_col=[0, 1, 2])
mi_image_mat_rasmi = mi_image_mat.copy()


In [23]:
mi_image_mat.index.get_level_values('Region').unique()
mi_rasmi.get('concrete').index.get_level_values('function').unique()

Index(['NR', 'RM', 'RS'], dtype='object', name='function')

In [24]:
# dictionary that maps RASMI R32 Regions to IMAGE R26 Regions

image_to_rasmi = {
    1: 'OECD_CAN',
    2: 'OECD_USA',
    3: 'LAM_MEX',
    4: 'LAM_LAM-L',
    5: 'LAM_BRA',
    6: 'LAM_LAM-L',
    7: ['MAF_MEA-H', 'MAF_MEA-M', 'MAF_NAF'],
    8: ['MAF_SSA-L', 'MAF_SSA-M'],
    9: ['MAF_SSA-L', 'MAF_SSA-M'],
    10: 'MAF_SAF',
    11: ['OECD_EFTA', 'OECD_EU12-H', 'OECD_EU12-M', 'OECD_EU15'],
    12: 'REF_EEU-FSU',
    13: 'OECD_TUR',
    14: ['REF_EEU-FSU', 'ASIA_TWN'],
    15: ['OECD_EEU', 'REF_CAS'],
    16: 'REF_RUS',
    17: ['MAF_MEA-H', 'MAF_MEA-M'],
    18: ['ASIA_IND', 'ASIA_OAS-CPA', 'ASIA_PAK'],
    19: 'OECD_KOR',
    20: 'ASIA_CHN',
    21: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    22: 'ASIA_IDN',
    23: 'OECD_JPN',
    24: 'OECD_AUNZ',
    25: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    26: ['MAF_SSA-L', 'MAF_SSA-M']
}

housing_type_image_to_rasmi = {
    1 : 'RS', #detached house to residential single family
    2 : 'RS', # semi detached house to residential single family
    3 : 'RM', # appartement to residential multi-family
    4 : 'RM', # high-rise to residential multi-family
}

housing_type_rasmi_to_image = {
    'RS' : [1, 2], # detached house and semi detached house to
    'RM' : [3, 4]  # appartement and high-rise to
}

materials_image_to_rasmi = {
    'Steel': 'steel',
    'Concrete': 'concrete',
    'Brick': 'brick',
    'Wood': 'wood',
    'Copper': 'copper',
    'Aluminium': 'aluminum',
    'Glass': 'glass'
    }
    

In [25]:
# Compare image mat 2020 with 2050 values
mi_mat_2020 = mi_image_mat.loc[2020]
mi_mat_2050 = mi_image_mat.loc[2050]

compare = mi_mat_2050 / mi_mat_2020 * 100
compare

Steel  Concrete   Wood  Copper  Aluminium  Glass  Brick
Region Building_type                                                         
1      1              100.0     100.0  100.0   100.0      100.0  100.0  100.0
       2              100.0     100.0  100.0   100.0      100.0  100.0  100.0
       3              100.0     100.0  100.0   100.0      100.0  100.0  100.0
       4              100.0     100.0  100.0   100.0      100.0  100.0  100.0
2      1              100.0     100.0  100.0   100.0      100.0  100.0  100.0
...                     ...       ...    ...     ...        ...    ...    ...
25     4              100.0     100.0  100.0   100.0      100.0  100.0  100.0
26     1              100.0     100.0  100.0   100.0      100.0  100.0  100.0
       2              100.0     100.0  100.0   100.0      100.0  100.0  100.0
       3              100.0     100.0  100.0   100.0      100.0  100.0  100.0
       4              100.0     100.0  100.0   100.0      100.0  100.0  100.0

[104 rows x 7 columns]

In [26]:
concrete_values = mi_rasmi.get("concrete").loc[:, "p_50"]

def replace_mat_mi_with_rasmi_mi(p_value: str, old_mat_mi: pd.DataFrame = mi_image_mat_rasmi) -> pd.DataFrame:

    for material_image, material_rasmi in materials_image_to_rasmi.items():
        intensities = mi_rasmi.get(material_rasmi).loc[:, p_value]

        for image_class, rasmi_region in image_to_rasmi.items():
            # convert str to list is necessary:
            if isinstance(rasmi_region, str):
                rasmi_region = [rasmi_region]

            # loop through IMAGE regions and get the mean concrete mi value for each region for RS and RM (housing types)
            for housingtype in set(housing_type_image_to_rasmi.values()):

                filtered = intensities[intensities.index.get_level_values('R5_32').isin(rasmi_region)
                                        & intensities.index.get_level_values('function').isin([housingtype])]

                mean = filtered.mean()
                mi_image_mat_rasmi.loc[([2020, 2050], image_class, housing_type_rasmi_to_image.get(housingtype)), material_image] = mean
            
    return mi_image_mat_rasmi


In [27]:
mi_replaced = replace_mat_mi_with_rasmi_mi('p_50')

In [28]:
# compare new material intensities with old ones
compare = mi_replaced/mi_image_mat*100
compare

Steel   Concrete        Wood       Copper  \
Year Region Building_type                                                   
2020 1      1              133.363428  66.655559   98.293925    10.571994   
            2              131.660399  47.532217  143.098190  1828.954950   
            3               38.970529  53.520607  124.048747    58.998547   
            4               46.622889  54.653272  110.918738  1828.954950   
     2      1              212.169562  73.032072   96.077087    14.515515   
...                               ...        ...         ...          ...   
2050 25     4               29.241110  84.413503   65.766909  1828.954950   
     26     1               91.393438  67.627628   94.595283    10.571994   
            2               90.670960  47.375109  143.556524  1828.954950   
            3               40.776794  55.509360  119.589854    58.998547   
            4               33.937670  60.736404   81.592417  1828.954950   

                            Aluminium       Glass       Brick  
Year Region Building_type                                      
2020 1      1                7.620139  112.804684   65.571741  
            2              212.701260  282.538834   79.672388  
            3               25.297151   46.708395  160.610009  
            4               22.307487  105.551000  321.220018  
     2      1               18.148148  133.529699   64.409164  
...                               ...         ...         ...  
2050 25     4               22.272219   43.179312  322.330698  
     26     1               13.785526  119.644100   72.806058  
            2              213.375967  299.669336   88.462383  
            3               25.257732   41.214854  169.848907  
            4               22.272727   59.211385  339.697813  

[208 rows x 7 columns]

In [29]:
# save as CSV 
mi_replaced.to_csv("Building_materials_rasmi.csv")